In [4]:
import warnings

warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning,
    message="This process .* is multi-threaded, use of fork.*"
)

In [ ]:
from sentence_transformers import SentenceTransformer

sentence_model_minilm = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

emb = sentence_model_minilm.encode(docs, show_progress_bar=True)

np.save("embeddings.npy", emb)
#emb = np.load("embeddings.npy")

In [10]:
vectorizer_model = CountVectorizer(
    stop_words = STOPWORDS,
    min_df=0.005,
    max_df=0.7,
    ngram_range=(1, 1)
)

In [ ]:
# ==== Coherence function ====
def compute_coherence(topic_model, docs):
    topics = [
        [word for word, _ in topic_model.get_topic(t)]
        for t in topic_model.get_topics().keys() if t != -1
    ]
    tokenized_docs = [doc.split() for doc in docs]
    dictionary = Dictionary(tokenized_docs)
    coherence_model = CoherenceModel(
        topics=topics,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence='c_v'
    )
    return coherence_model.get_coherence()


# ==== Topic Diversity ====
def compute_topic_diversity(topic_model, topk=10):
    topics = [
        [word for word, _ in topic_model.get_topic(t)[:topk]]
        for t in topic_model.get_topics().keys()
        if t != -1
    ]
    all_words = [word for topic in topics for word in topic]
    unique_words = set(all_words)
    diversity = len(unique_words) / len(all_words) if len(all_words) > 0 else 0
    return diversity


def build_topic_model(n_neighbors, n_components, min_cluster_size=10, min_dist=0.0):
    # UMAP
    umap_model = umap.UMAP(
        n_neighbors=n_neighbors,
        n_components=n_components,
        min_dist=min_dist,
        metric='cosine',
        random_state=42
    )

    # HDBSCAN
    hdbscan_model = HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=None,
        metric='euclidean',
        cluster_selection_method='eom'
    )

    # BERTopic
    return BERTopic(
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        top_n_words=10,
        nr_topics=None,  # on laisse BERTopic choisir
        verbose=False
    )



In [ ]:
from itertools import product

n_components_list = [5,7,10]
n_neighbors_list = range(18,20)
min_cluster_size_list = range(18,20)
min_dist_list = [0.3,0.5] 

results = []
models = {}

# ==== Grid Search ====
for n_comp, n_nbr, min_size, min_d in product(
        n_components_list, n_neighbors_list, min_cluster_size_list, min_dist_list):

    # Model    
    tm = build_topic_model(
        n_neighbors=n_nbr,
        n_components=n_comp,
        min_cluster_size=min_size,
        min_dist=min_d
    )
    
    # Fit & transform
    topics, probs = tm.fit_transform(docs, emb)
    
    # Récupérer le count du topic 0
    topic_info = tm.get_topic_info()
    topic0_count = topic_info.loc[topic_info['Topic'] == 0, 'Count'].values[0] if 0 in topic_info['Topic'].values else 0
    
    # Metrics
    coh = compute_coherence(tm, docs)
    div = compute_topic_diversity(tm)
    
    # Score
    score = coh * div
    
    # Save
    key = (n_comp, n_nbr, min_size, min_d)
    models[key] = (tm, topics, probs)
    
    results.append({
        "n_components": n_comp,
        "n_neighbors": n_nbr,
        "min_cluster_size": min_size,
        "min_dist": min_d,
        "coherence": coh,
        "diversity": div,
        "score": score,
        "n_topics": len(set(topics)) - (1 if -1 in topics else 0),  # nombre de topics réels
        "topic0_count": topic0_count
    })
    
    print(f"comp={n_comp}, neigh={n_nbr}, min_size={min_size}, min_dist={min_d} | "
          f"coh={coh:.4f}, div={div:.4f}, score={score:.4f}, n_topics={len(set(topics))}, "
          f"topic0_count={topic0_count}")

comp=5, neigh=18, min_size=18, min_dist=0.3 | coh=0.4066, div=0.8500, score=0.3456, n_topics=6, topic0_count=4716
comp=5, neigh=18, min_size=18, min_dist=0.5 | coh=0.3647, div=0.7833, score=0.2857, n_topics=7, topic0_count=4716
comp=5, neigh=18, min_size=19, min_dist=0.3 | coh=0.4066, div=0.8500, score=0.3456, n_topics=6, topic0_count=4716
comp=5, neigh=18, min_size=19, min_dist=0.5 | coh=0.3647, div=0.7833, score=0.2857, n_topics=7, topic0_count=4716
comp=5, neigh=19, min_size=18, min_dist=0.3 | coh=0.3647, div=0.7833, score=0.2857, n_topics=7, topic0_count=4716
comp=5, neigh=19, min_size=18, min_dist=0.5 | coh=0.3487, div=0.8000, score=0.2790, n_topics=7, topic0_count=4716
comp=5, neigh=19, min_size=19, min_dist=0.3 | coh=0.3647, div=0.7833, score=0.2857, n_topics=7, topic0_count=4716
comp=5, neigh=19, min_size=19, min_dist=0.5 | coh=0.3487, div=0.8000, score=0.2790, n_topics=7, topic0_count=4716
comp=7, neigh=18, min_size=18, min_dist=0.3 | coh=0.3701, div=0.7833, score=0.2899, n_to

In [ ]:
df_grid = pd.DataFrame(results)
df_grid = df_grid.sort_values(["coherence"], ascending=False).reset_index(drop=True)

print(df_grid.head(10))

df_grid.to_csv("grid_search_bertopic4.csv", sep=";", index=False)

   n_components  n_neighbors  min_cluster_size  min_dist  coherence  \
0             7           19                18       0.3   0.432184   
1             7           19                19       0.3   0.432184   
2            10           19                19       0.3   0.426228   
3            10           19                18       0.3   0.426228   
4             5           18                18       0.3   0.406571   
5             5           18                19       0.3   0.406571   
6             7           19                18       0.5   0.385524   
7             7           19                19       0.5   0.385524   
8             7           18                19       0.3   0.384330   
9            10           19                19       0.5   0.382522   

   diversity     score  n_topics  topic0_count  
0   0.866667  0.374559         6          4716  
1   0.866667  0.374559         6          4716  
2   0.866667  0.369397         6          4716  
3   0.866667  0.369397